# Fine-tuning Qwen for Maths

In [ ]:
%%capture
!pip install -q --no-cache-dir unsloth
!pip  install -q --no-cache-dir --upgrade bitsandbytes datasets accelerate trl peft

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel
import torch
import gc

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    low_cpu_mem_usage = True
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-14B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.10 patched 48 layers with 48 QKV layers, 48 O layers and 48 MLP layers.


In [ ]:
from trl import SFTTrainer, SFTConfig
import torch
import gc

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = True, # FIXED: Enforces sequence packing to align with Unsloth's padding_free layout
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "no",  # CRITICAL: Blocks intermediate back-and-forth CPU RAM checkpoints
        eval_strategy = "no",
        packing = True         # FIXED: Confirms packing behavior inside the TRL configuration schema
    ),
)

# Start training loop
trainer_stats = trainer.train()

# Final post-training cleanup
gc.collect()
torch.cuda.empty_cache()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2999 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/2999 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,154 | Num Epochs = 1 | Total steps = 145
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.689350
20,0.448985
30,0.390795
40,0.382384
50,0.376936
60,0.355402
70,0.375650
80,0.362852
90,0.352360
100,0.374176


In [ ]:
import json
import urllib.request
import random
from datasets import Dataset


urls = [
    "https://raw.githubusercontent.com/JIA-Lab-research/MR-GSM8K/master/dataset/MR-GSM8k.json",
]

raw_data = None
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            raw_data = json.loads(response.read().decode('utf-8'))

if raw_data is None:
    raise RuntimeError("404 Error Avoided: All known remote paths failed. Ensure internet access is enabled in the right panel of your Kaggle notebook.")


random.seed(42)
sampled_data = random.sample(raw_data, min(6000, len(raw_data)))


dataset_dict = {
    "question": [item["question"] for item in sampled_data],
    "ground_truth_solution": [item["ground_truth_solution"] for item in sampled_data]
}
dataset = Dataset.from_dict(dataset_dict)


TOOL_GATE_SYSTEM = (
    "You decide whether a calculation tool is needed to answer a multiple-choice "
    "question.\n\n"
    "Call a tool ONLY if answering requires exact computation you cannot reliably "
    "do mentally: large-number arithmetic, factorials, combinations/permutations, "
    "solving equations, integrals/derivatives, gcd/lcm/prime factorization, "
    "compound interest, or unit/weight conversion.\n\n"
    "Do NOT call a tool for conceptual, definitional, or reasoning questions "
    "(abstract algebra theory, statistics concepts, 'which statement is true', "
    "factual recall). For those, output the single word: NONE\n\n"
    "If a tool is needed, emit exactly one tool call and nothing else."
)

def format_prompt(examples):
    questions = examples["question"]
    solutions = examples["ground_truth_solution"]

    formatted_texts = []
    for q, sol in zip(questions, solutions):
        clean_code = str(sol).replace('\n', ' ').replace('"', "'")
        tool_call = f'call:compute_math{{expression:"{clean_code}"}}'

        text = (
            f"<|im_start|>system\n{TOOL_GATE_SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{q}<|im_end|>\n"
            f"<|im_start|>assistant\n{tool_call}<|im_end|>"
        )
        formatted_texts.append(text)

    return {"text": formatted_texts}


dataset = dataset.map(format_prompt, batched=True, num_proc=1)


gc.collect()
torch.cuda.empty_cache()

Successfully connected and downloaded dataset from: https://raw.githubusercontent.com/JIA-Lab-research/MR-GSM8K/master/dataset/MR-GSM8K.json


Map (num_proc=1):   0%|          | 0/2999 [00:00<?, ? examples/s]

In [ ]:
import os
import json
from kaggle_secrets import UserSecretsClient

model_dir = "/kaggle/working/qwen-math-tool-router"
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

user_secrets = UserSecretsClient()
username = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_KEY")

metadata = {
    "title": "Qwen 2.5 14B Math Tool Router2-0",
    "id": f"{username}/qwen-math-tool-router2-0",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(f"{model_dir}/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Publishing model weights folder publicly to Kaggle under: {username}/qwen-math-tool-router2-0")


!kaggle datasets create -p {model_dir} -u --dir-mode zip

Publishing model weights folder publicly to Kaggle under: fidelguri/qwen-math-tool-router2-0
Starting upload for file tokenizer_config.json
100%|██████████████████████████████████████| 4.46k/4.46k [00:00<00:00, 11.6kB/s]
Upload successful: tokenizer_config.json (4KB)
Starting upload for file adapter_model.safetensors
100%|█████████████████████████████████████████| 263M/263M [00:02<00:00, 100MB/s]
Upload successful: adapter_model.safetensors (263MB)
Starting upload for file README.md
100%|██████████████████████████████████████| 5.13k/5.13k [00:00<00:00, 13.6kB/s]
Upload successful: README.md (5KB)
Starting upload for file chat_template.jinja
100%|██████████████████████████████████████| 2.45k/2.45k [00:00<00:00, 5.41kB/s]
Upload successful: chat_template.jinja (2KB)
Starting upload for file adapter_config.json
100%|██████████████████████████████████████| 1.22k/1.22k [00:00<00:00, 3.34kB/s]
Upload successful: adapter_config.json (1KB)
Starting upload for file tokenizer.json
100%|█████████

# Results tested with our main pipeline

In [ ]:
df = online_evaluate(model, client, strategy = P0_BASELINE, competitions=(3,), max_questions=500, use_rag=False)
report_results(df)


--- Starting Comp 3 (Maths) ---


Comp 3: 100%|██████████| 500/500 [2:42:28<00:00, 19.50s/it]  

  ✓ Saved 500 rows -> online_eval_20260602_113742_through_comp3.csv
RESULTS — Qwen/Qwen2.5-14B-Instruct

Accuracy per competition (online, biased toward early levels):
                    n    acc  timeouts
competition_name                      
Maths             500  0.526        99

Accuracy per level (note: smaller n at higher levels):
         n       acc
level               
1      238  0.563025
2      133  0.481203
3       64  0.500000
4       32  0.656250
5       21  0.333333
6        7  0.571429
7        4  0.250000
8        1  0.000000

Accuracy per competition × level:
competition_name  Maths
level                  
1                 0.563
2                 0.481
3                 0.500
4                 0.656
5                 0.333
6                 0.571
7                 0.250
8                 0.000

Latency per competition (s):
                    n  mean_total  mean_model  mean_rag  max_total
competition_name                                                  
Maths     